# Erro Percentual e Visualização — Método Geométrico

## Importações

In [1]:
import pandas as pd
from pathlib import Path
import numpy as np
import plotly.graph_objects as go

from utilitarios_populacional import (
    COLUNA_MUNICIPIO,
    COLUNA_CODIGO_IBGE,
    COLUNA_REGIONAL,
    normalizar_codigo_ibge,
    calcular_taxa_geometrica,
    estimar_populacao_geometrica,
    calcular_taxa_exponencial,
    estimar_populacao_exponencial,
    resumir_ep,
    agregar_ep_estadual,
)

## Configuração

In [2]:
CAMINHO_ARQUIVO_ENTRADA = Path('/home/anna/tcc/Seleção e Transformação/População - Faixa Etária/tabela_populacao_faixa_etaria.csv')
CAMINHO_ARQUIVO_SAIDA = Path('/home/anna/tcc/Mineração de Dados - Avaliação e Apresentação/Análise Populacional/populacao_interpolacao_extrapolacao_geometrica.csv')
CAMINHO_REGIONAIS = Path("/home/anna/tcc/Seleção e Transformação/Municípios/tabela_municipios_regionais.csv")

LISTA_ANOS_CENSITARIOS = [1970, 1980, 1991, 2000, 2010, 2022]
ANO_INICIO = 1970
ANO_FIM = 2025
LISTA_ANOS_COMPLETOS = list(range(ANO_INICIO, ANO_FIM + 1))
ANOS_A_EXTRAPOLAR = [ano for ano in LISTA_ANOS_COMPLETOS if ano > LISTA_ANOS_CENSITARIOS[-1]]

FIG_WIDTH, FIG_HEIGHT = 1100, 550
FONTE_FAMILIA = "Arial"
FONTE_TITULO = 18
FONTE_EIXO = 13

COR_CENSO = "#1B1B1B"
COR_INTERPOLACAO = "#2E5C8A"
COR_EXTRAPOLACAO = "#C0392B"
COR_EP_INTERPOLACAO = "#2E5C8A"
COR_EP_EXTRAPOLACAO = "#C0392B"
COR_LIMITE_10PCT = "#999999"
COR_BARRAS_REGIONAL = "#2E5C8A"

TITULO_GRAFICO_POPULACAO = "População Total de Minas Gerais — Interpolação e Extrapolação (Método Geométrico)"
TITULO_GRAFICO_ERRO = "Erro Percentual (EP) do Estado — Anos Censitários Testados"
TITULO_GRAFICO_REGIONAL = "População Feminina por Regional de Saúde — Minas Gerais"

EIXO_X_POPULACAO = "Ano"
EIXO_Y_POPULACAO = "População (Minas Gerais)"
EIXO_X_ERRO = "Ano testado"
EIXO_Y_ERRO = "Erro Percentual — EP (%)"
EIXO_X_REGIONAL = "Regional de Saúde"
EIXO_Y_REGIONAL = "População Feminina"

ANO_REFERENCIA_GRAFICO_REGIONAL = 2025

## Carregamento dos Dados

In [3]:
tabela_populacao = pd.read_csv(CAMINHO_ARQUIVO_ENTRADA)
tabela_populacao[COLUNA_CODIGO_IBGE] = normalizar_codigo_ibge(tabela_populacao[COLUNA_CODIGO_IBGE])
lista_colunas_totais = [f"TOTAL_{ano}" for ano in LISTA_ANOS_CENSITARIOS]
tabela_totais = tabela_populacao[[COLUNA_MUNICIPIO, COLUNA_CODIGO_IBGE] + lista_colunas_totais].copy()

tabela_saida = pd.read_csv(CAMINHO_ARQUIVO_SAIDA)
tabela_saida[COLUNA_CODIGO_IBGE] = normalizar_codigo_ibge(tabela_saida[COLUNA_CODIGO_IBGE])

tabela_regionais = pd.read_csv(CAMINHO_REGIONAIS)
tabela_regionais[COLUNA_CODIGO_IBGE] = normalizar_codigo_ibge(tabela_regionais[COLUNA_CODIGO_IBGE])

codigos_saida_bruto = set(tabela_saida[COLUNA_CODIGO_IBGE])
codigos_regionais_bruto = set(tabela_regionais[COLUNA_CODIGO_IBGE])
diferenca = codigos_saida_bruto - codigos_regionais_bruto
if diferenca:
    print(f"ATENÇÃO: {len(diferenca)} código(s) IBGE de tabela_saida sem correspondência em tabela_regionais: {sorted(diferenca, key=str)}")
else:
    print("Todos os municípios de tabela_saida têm correspondência em tabela_regionais.")

mapa_regional = tabela_regionais.set_index(COLUNA_CODIGO_IBGE)[COLUNA_REGIONAL]

print(f"tabela_totais    : {tabela_totais.shape[0]} municípios")
print(f"tabela_saida     : {tabela_saida.shape[0]} municípios, {ANO_INICIO}-{ANO_FIM}")
print(f"tabela_regionais : {tabela_regionais.shape[0]} municípios mapeados")

Todos os municípios de tabela_saida têm correspondência em tabela_regionais.
tabela_totais    : 853 municípios
tabela_saida     : 853 municípios, 1970-2025
tabela_regionais : 853 municípios mapeados


## Erro Percentual da Interpolação

In [4]:
CENSOS_DO_MEIO = LISTA_ANOS_CENSITARIOS[1:-1]  # [1980, 1991, 2000, 2010]

lista_ep_interpolacao = []

for ano_escondido in CENSOS_DO_MEIO:
    censos_restantes = [ano for ano in LISTA_ANOS_CENSITARIOS if ano != ano_escondido]
    ano_anterior = max(ano for ano in censos_restantes if ano < ano_escondido)
    ano_posterior = min(ano for ano in censos_restantes if ano > ano_escondido)

    populacao_anterior = tabela_totais[f"TOTAL_{ano_anterior}"]
    populacao_posterior = tabela_totais[f"TOTAL_{ano_posterior}"]
    tempo_total = ano_posterior - ano_anterior
    tempo_decorrido = ano_escondido - ano_anterior

    r_g = calcular_taxa_geometrica(populacao_anterior, populacao_posterior, tempo_total)
    populacao_estimada = estimar_populacao_geometrica(populacao_anterior, r_g, tempo_decorrido)
    populacao_observada = tabela_totais[f"TOTAL_{ano_escondido}"]

    ep = ((populacao_estimada - populacao_observada) / populacao_observada) * 100

    lista_ep_interpolacao.append(pd.DataFrame({
        COLUNA_MUNICIPIO: tabela_totais[COLUNA_MUNICIPIO],
        COLUNA_CODIGO_IBGE: tabela_totais[COLUNA_CODIGO_IBGE],
        "ANO_TESTADO": ano_escondido,
        "TIPO": "Interpolação",
        "CENSOS_USADOS": f"{ano_anterior}-{ano_posterior}",
        "POPULACAO_OBSERVADA": populacao_observada,
        "POPULACAO_ESTIMADA": populacao_estimada,
        "EP": ep,
    }))

tabela_ep_interpolacao = pd.concat(lista_ep_interpolacao, ignore_index=True)
tabela_ep_interpolacao.head()


,MUNICÍPIOS,CÓDIGO IBGE,ANO_TESTADO,TIPO,CENSOS_USADOS,POPULACAO_OBSERVADA,POPULACAO_ESTIMADA,EP
0,abadia dos dourados,3100104,1980,Interpolação,1970-1991,3936,4008.532720,1.842803
1,abaete,3100203,1980,Interpolação,1970-1991,9926,10373.066479,4.503994
2,abre campo,3100302,1980,Interpolação,1970-1991,9375,9464.885168,0.958775
3,acaiaca,3100401,1980,Interpolação,1970-1991,1822,1808.447373,-0.743832
4,aguanil,3100807,1980,Interpolação,1970-1991,1560,1431.690863,-8.224945


## Erro Percentual da Extrapolação (Exponencial)

In [5]:
PARES_EXTRAPOLACAO_TESTE = [
    (1991, 2000, 2010),
    (2000, 2010, 2022),
]

lista_ep_extrapolacao = []

for ano_inicial, ano_final, ano_alvo in PARES_EXTRAPOLACAO_TESTE:
    populacao_inicial = tabela_totais[f"TOTAL_{ano_inicial}"]
    populacao_final = tabela_totais[f"TOTAL_{ano_final}"]
    tempo_total = ano_final - ano_inicial
    tempo_decorrido = ano_alvo - ano_final

    k_g = calcular_taxa_exponencial(populacao_inicial, populacao_final, tempo_total)
    populacao_estimada = estimar_populacao_exponencial(populacao_final, k_g, tempo_decorrido)
    populacao_observada = tabela_totais[f"TOTAL_{ano_alvo}"]

    ep = ((populacao_estimada - populacao_observada) / populacao_observada) * 100

    lista_ep_extrapolacao.append(pd.DataFrame({
        COLUNA_MUNICIPIO: tabela_totais[COLUNA_MUNICIPIO],
        COLUNA_CODIGO_IBGE: tabela_totais[COLUNA_CODIGO_IBGE],
        "ANO_TESTADO": ano_alvo,
        "TIPO": "Extrapolação",
        "CENSOS_USADOS": f"{ano_inicial}-{ano_final}",
        "POPULACAO_OBSERVADA": populacao_observada,
        "POPULACAO_ESTIMADA": populacao_estimada,
        "EP": ep,
    }))

tabela_ep_extrapolacao = pd.concat(lista_ep_extrapolacao, ignore_index=True)
tabela_ep_extrapolacao.head()

,MUNICÍPIOS,CÓDIGO IBGE,ANO_TESTADO,TIPO,CENSOS_USADOS,POPULACAO_OBSERVADA,POPULACAO_ESTIMADA,EP
0,abadia dos dourados,3100104,2010,Extrapolação,1991-2000,3676,2949.446810,-19.764777
1,abaete,3100203,2010,Extrapolação,1991-2000,11171,11393.656393,1.993164
2,abre campo,3100302,2010,Extrapolação,1991-2000,7238,4616.297734,-36.221363
3,acaiaca,3100401,2010,Extrapolação,1991-2000,1881,1808.435846,-3.857743
4,aguanil,3100807,2010,Extrapolação,1991-2000,1938,1973.075974,1.809906


## Resumo do Erro

In [6]:
tabela_resumo_ep = pd.DataFrame([
    resumir_ep(tabela_ep_interpolacao, "Interpolação"),
    resumir_ep(tabela_ep_extrapolacao, "Extrapolação"),
])

tabela_resumo_ep

,Validação,EP médio (%),EP mediano (%),MAPE (%),Desvio-padrão (%)
0,Interpolação,-2.245,-0.703,9.488,16.556
1,Extrapolação,63.419,-0.691,79.871,234.179


## Agregação Estadual — Erro

In [7]:
tabela_ep_estadual = pd.concat([
    agregar_ep_estadual(tabela_ep_interpolacao),
    agregar_ep_estadual(tabela_ep_extrapolacao),
], ignore_index=True).sort_values("ANO_TESTADO")

tabela_ep_estadual

,ANO_TESTADO,TIPO,POPULACAO_OBSERVADA,POPULACAO_ESTIMADA,EP_ESTADO
0,1980,Interpolação,7338100,6.686035e+06,-8.886019
1,1991,Interpolação,8279567,8.020464e+06,-3.129423
2,2000,Interpolação,9255322,8.667268e+06,-6.353688
3,2010,Interpolação,9236906,9.782865e+06,5.910626
4,2010,Extrapolação,9236906,1.339105e+07,44.973321
5,2022,Extrapolação,10524280,9.262059e+06,-11.993423


## Agregação Estadual — População

In [8]:
colunas_ano_completas = [f"TOTAL_{ano}" for ano in LISTA_ANOS_COMPLETOS]
populacao_estadual_completa = tabela_saida[colunas_ano_completas].sum()
populacao_estadual_completa.index = LISTA_ANOS_COMPLETOS

populacao_estadual_censos = tabela_totais[lista_colunas_totais].sum()
populacao_estadual_censos.index = LISTA_ANOS_CENSITARIOS

populacao_estadual_censos


1970     5772606
1980     7339191
1991     8346023
2000     9255322
2010     9236906
2022    10524280
dtype: int64

## População por Regional de Saúde

In [9]:
tabela_saida_com_regional = tabela_saida.copy()
tabela_saida_com_regional[COLUNA_REGIONAL] = tabela_saida_com_regional[COLUNA_CODIGO_IBGE].map(mapa_regional)

colunas_totais_completas = [f"TOTAL_{ano}" for ano in LISTA_ANOS_COMPLETOS]

tabela_populacao_por_regional = (
    tabela_saida_com_regional
    .groupby(COLUNA_REGIONAL, as_index=False)[colunas_totais_completas]
    .sum()
)

tabela_populacao_por_regional.head()

,REGIONAL NORMALIZADA,TOTAL_1970,TOTAL_1971,TOTAL_1972,TOTAL_1973,TOTAL_1974,TOTAL_1975,TOTAL_1976,TOTAL_1977,TOTAL_1978,...,TOTAL_2016,TOTAL_2017,TOTAL_2018,TOTAL_2019,TOTAL_2020,TOTAL_2021,TOTAL_2022,TOTAL_2023,TOTAL_2024,TOTAL_2025
0,alfenas,139443,142504.0,145634.0,148843.0,152144.0,155533.0,159015.0,162593.0,166266.0,...,223557,225492,227459,229444,231457,233497,235547,237650,239767,241913
1,barbacena,156471,159877.0,163402.0,167072.0,170895.0,174875.0,179021.0,183344.0,187849.0,...,247024,249320,251668,254057,256501,258991,261518,264128,266775,269476
2,belo horizonte,951683,998833.0,1049124.0,1102870.0,1160398.0,1222092.0,1288380.0,1359755.0,1436788.0,...,2544583,2583253,2622809,2663270,2704667,2747021,2790345,2834729,2880137,2926635
3,coronel fabriciano,218147,223776.0,229983.0,236845.0,244427.0,252809.0,262083.0,272338.0,283695.0,...,385416,388550,391804,395177,398675,402305,406050,409970,414019,418217
4,diamantina,173917,175594.0,177284.0,178998.0,180746.0,182522.0,184329.0,186165.0,188029.0,...,200029,199662,199324,199019,198742,198496,198264,198094,197939,197807


## Gráfico 1 — Curva da População do Estado

In [10]:
anos_interpolados = [ano for ano in LISTA_ANOS_COMPLETOS if ano <= LISTA_ANOS_CENSITARIOS[-1]]
populacao_interpolada = [populacao_estadual_completa[ano] for ano in anos_interpolados]

anos_extrapolados = [ano for ano in LISTA_ANOS_COMPLETOS if ano >= LISTA_ANOS_CENSITARIOS[-1]]
populacao_extrapolada = [populacao_estadual_completa[ano] for ano in anos_extrapolados]

populacao_censos = [populacao_estadual_censos[ano] for ano in LISTA_ANOS_CENSITARIOS]
populacao_extrapolados_destaque = [populacao_estadual_completa[ano] for ano in ANOS_A_EXTRAPOLAR]

fig_populacao = go.Figure()

fig_populacao.add_trace(go.Scatter(
    x=anos_interpolados, y=populacao_interpolada,
    mode="lines", name="Interpolação (entre censos)",
    line=dict(color=COR_INTERPOLACAO, width=2.5),
    hovertemplate="Ano: %{x}<br>População: %{y:,.0f}<extra></extra>",
))

fig_populacao.add_trace(go.Scatter(
    x=anos_extrapolados, y=populacao_extrapolada,
    mode="lines", name="Extrapolação (após o último censo)",
    line=dict(color=COR_EXTRAPOLACAO, width=2.5, dash="dash"),
    hovertemplate="Ano: %{x}<br>População: %{y:,.0f}<extra></extra>",
))

fig_populacao.add_trace(go.Scatter(
    x=LISTA_ANOS_CENSITARIOS, y=populacao_censos,
    mode="markers", name="Censos (observado)",
    marker=dict(color=COR_CENSO, size=11),
    hovertemplate="Censo %{x}<br>População: %{y:,.0f}<extra></extra>",
))

fig_populacao.add_trace(go.Scatter(
    x=ANOS_A_EXTRAPOLAR, y=populacao_extrapolados_destaque,
    mode="markers", name="Anos extrapolados",
    marker=dict(color=COR_EXTRAPOLACAO, size=9, symbol="diamond"),
    hovertemplate="Ano: %{x}<br>População: %{y:,.0f}<extra></extra>",
))

fig_populacao.update_layout(
    title=dict(text=TITULO_GRAFICO_POPULACAO, x=0.5, xanchor="center",
               font=dict(size=FONTE_TITULO, family=FONTE_FAMILIA)),
    xaxis_title=EIXO_X_POPULACAO, yaxis_title=EIXO_Y_POPULACAO,
    font=dict(family=FONTE_FAMILIA, size=FONTE_EIXO),
    width=FIG_WIDTH, height=FIG_HEIGHT,
    hovermode="x unified",
    legend=dict(orientation="h", y=-0.22, x=0.5, xanchor="center"),
    plot_bgcolor="white", paper_bgcolor="white",
)

fig_populacao.update_xaxes(
    showgrid=True, gridcolor="#E5E5E5",
    tickmode="array",
    tickvals=sorted(set(LISTA_ANOS_CENSITARIOS + ANOS_A_EXTRAPOLAR)),
    tickangle=-45,
)
fig_populacao.update_yaxes(showgrid=True, gridcolor="#E5E5E5", tickformat=",.0f")

fig_populacao.show()


## Gráfico 2 — Erro Percentual do Estado

In [11]:
fig_erro = go.Figure()

for tipo, cor in [("Interpolação", COR_EP_INTERPOLACAO), ("Extrapolação", COR_EP_EXTRAPOLACAO)]:
    subset = tabela_ep_estadual[tabela_ep_estadual["TIPO"] == tipo]
    fig_erro.add_trace(go.Bar(
        x=subset["ANO_TESTADO"], y=subset["EP_ESTADO"],
        name=tipo, marker=dict(color=cor),
        text=[f"{valor:.2f}%" for valor in subset["EP_ESTADO"]],
        textposition="outside",
        hovertemplate="Ano testado: %{x}<br>EP: %{y:.2f}%<extra></extra>",
    ))

fig_erro.add_hline(y=10, line=dict(color=COR_LIMITE_10PCT, width=1, dash="dot"),
                    annotation_text="+10%", annotation_position="top right")
fig_erro.add_hline(y=-10, line=dict(color=COR_LIMITE_10PCT, width=1, dash="dot"),
                    annotation_text="-10%", annotation_position="bottom right")
fig_erro.add_hline(y=0, line=dict(color="#444444", width=1))

fig_erro.update_layout(
    title=dict(text=TITULO_GRAFICO_ERRO, x=0.5, xanchor="center",
               font=dict(size=FONTE_TITULO, family=FONTE_FAMILIA)),
    xaxis_title=EIXO_X_ERRO, yaxis_title=EIXO_Y_ERRO,
    font=dict(family=FONTE_FAMILIA, size=FONTE_EIXO),
    width=FIG_WIDTH, height=FIG_HEIGHT,
    barmode="group",
    legend=dict(orientation="h", y=-0.22, x=0.5, xanchor="center"),
    plot_bgcolor="white", paper_bgcolor="white",
)

fig_erro.update_xaxes(
    showgrid=True, gridcolor="#E5E5E5",
    tickmode="array",
    tickvals=sorted(tabela_ep_estadual["ANO_TESTADO"].unique()),
)
fig_erro.update_yaxes(showgrid=True, gridcolor="#E5E5E5")

fig_erro.show()


## Gráfico 3 — População por Regional de Saúde

In [12]:
coluna_ano_referencia = f"TOTAL_{ANO_REFERENCIA_GRAFICO_REGIONAL}"

tabela_grafico_regional = tabela_populacao_por_regional[[COLUNA_REGIONAL, coluna_ano_referencia]].copy()
tabela_grafico_regional = tabela_grafico_regional.sort_values(coluna_ano_referencia, ascending=False)

nomes_das_regionais = [nome.title() for nome in tabela_grafico_regional[COLUNA_REGIONAL]]

fig_populacao_regional = go.Figure()

fig_populacao_regional.add_trace(go.Bar(
    x=nomes_das_regionais,
    y=tabela_grafico_regional[coluna_ano_referencia],
    marker=dict(color=COR_BARRAS_REGIONAL),
    text=[f"{valor:,.0f}" for valor in tabela_grafico_regional[coluna_ano_referencia]],
    textposition="outside",
    hovertemplate="%{x}<br>População: %{y:,.0f}<extra></extra>",
))

fig_populacao_regional.update_layout(
    title=dict(text=f"{TITULO_GRAFICO_REGIONAL} ({ANO_REFERENCIA_GRAFICO_REGIONAL})", x=0.5, xanchor="center",
               font=dict(size=FONTE_TITULO, family=FONTE_FAMILIA)),
    xaxis_title=EIXO_X_REGIONAL, yaxis_title=EIXO_Y_REGIONAL,
    font=dict(family=FONTE_FAMILIA, size=FONTE_EIXO),
    width=FIG_WIDTH + 200, height=FIG_HEIGHT,
    plot_bgcolor="white", paper_bgcolor="white",
)

fig_populacao_regional.update_xaxes(tickangle=-45)
fig_populacao_regional.update_yaxes(showgrid=True, gridcolor="#E5E5E5", tickformat=",.0f")

fig_populacao_regional.show()


## Erro Percentual Anualizado

O EP do Gráfico 2 mistura testes com janelas de tempo bem diferentes (9 a 22 anos entre os censos usados como referência), o que enviesa a comparação — um teste com janela maior tende a acumular mais erro só por cobrir mais tempo, não porque o método seja pior. O EP anualizado divide o EP pelo número de anos decorridos em cada teste, dando uma taxa de erro **por ano**, comparável entre janelas de tamanhos diferentes.

$$EP_{anual} = \frac{EP_{estado}}{t_{decorrido}}$$

Para a interpolação, $t_{decorrido}$ é o tempo entre o censo anterior e o ano escondido (o mesmo expoente usado em `estimar_populacao_geometrica`). Para a extrapolação, é o tempo entre o último censo conhecido e o ano-alvo (o mesmo expoente usado em `estimar_populacao_exponencial`).

In [13]:
mapa_tempo_decorrido_interpolacao = {}
for ano_escondido in CENSOS_DO_MEIO:
    censos_restantes = [ano for ano in LISTA_ANOS_CENSITARIOS if ano != ano_escondido]
    ano_anterior = max(ano for ano in censos_restantes if ano < ano_escondido)
    mapa_tempo_decorrido_interpolacao[ano_escondido] = ano_escondido - ano_anterior

mapa_tempo_decorrido_extrapolacao = {}
for ano_inicial, ano_final, ano_alvo in PARES_EXTRAPOLACAO_TESTE:
    mapa_tempo_decorrido_extrapolacao[ano_alvo] = ano_alvo - ano_final

lista_tempo_decorrido = []
for _, linha in tabela_ep_estadual.iterrows():
    if linha["TIPO"] == "Interpolação":
        lista_tempo_decorrido.append(mapa_tempo_decorrido_interpolacao[linha["ANO_TESTADO"]])
    else:
        lista_tempo_decorrido.append(mapa_tempo_decorrido_extrapolacao[linha["ANO_TESTADO"]])

tabela_ep_estadual_anualizado = tabela_ep_estadual.copy()
tabela_ep_estadual_anualizado["TEMPO_DECORRIDO"] = lista_tempo_decorrido
tabela_ep_estadual_anualizado["EP_ANUALIZADO"] = (
    tabela_ep_estadual_anualizado["EP_ESTADO"] / tabela_ep_estadual_anualizado["TEMPO_DECORRIDO"]
)

tabela_ep_estadual_anualizado

,ANO_TESTADO,TIPO,POPULACAO_OBSERVADA,POPULACAO_ESTIMADA,EP_ESTADO,TEMPO_DECORRIDO,EP_ANUALIZADO
0,1980,Interpolação,7338100,6.686035e+06,-8.886019,10,-0.888602
1,1991,Interpolação,8279567,8.020464e+06,-3.129423,11,-0.284493
2,2000,Interpolação,9255322,8.667268e+06,-6.353688,9,-0.705965
3,2010,Interpolação,9236906,9.782865e+06,5.910626,10,0.591063
4,2010,Extrapolação,9236906,1.339105e+07,44.973321,10,4.497332
5,2022,Extrapolação,10524280,9.262059e+06,-11.993423,12,-0.999452


## Gráfico 4 — Erro Percentual Anualizado do Estado

In [14]:
fig_erro_anualizado = go.Figure()

for tipo, cor in [("Interpolação", COR_EP_INTERPOLACAO), ("Extrapolação", COR_EP_EXTRAPOLACAO)]:
    subset = tabela_ep_estadual_anualizado[tabela_ep_estadual_anualizado["TIPO"] == tipo]
    fig_erro_anualizado.add_trace(go.Bar(
        x=subset["ANO_TESTADO"], y=subset["EP_ANUALIZADO"],
        name=tipo, marker=dict(color=cor),
        text=[f"{valor:.3f}%" for valor in subset["EP_ANUALIZADO"]],
        textposition="outside",
        hovertemplate="Ano testado: %{x}<br>EP anualizado: %{y:.3f}%/ano<extra></extra>",
    ))

fig_erro_anualizado.add_hline(y=0, line=dict(color="#444444", width=1))

fig_erro_anualizado.update_layout(
    title=dict(text="Erro Percentual Anualizado do Estado — Anos Censitários Testados", x=0.5, xanchor="center",
               font=dict(size=FONTE_TITULO, family=FONTE_FAMILIA)),
    xaxis_title=EIXO_X_ERRO, yaxis_title="Erro Percentual Anualizado (%/ano)",
    font=dict(family=FONTE_FAMILIA, size=FONTE_EIXO),
    width=FIG_WIDTH, height=FIG_HEIGHT,
    barmode="group",
    legend=dict(orientation="h", y=-0.22, x=0.5, xanchor="center"),
    plot_bgcolor="white", paper_bgcolor="white",
)

fig_erro_anualizado.update_xaxes(
    showgrid=True, gridcolor="#E5E5E5",
    tickmode="array",
    tickvals=sorted(tabela_ep_estadual_anualizado["ANO_TESTADO"].unique()),
)
fig_erro_anualizado.update_yaxes(showgrid=True, gridcolor="#E5E5E5")

fig_erro_anualizado.show()